# 03 · Explainability, fairness audit and failure analysis
Requires artifacts from notebook 02 (or `python -m src.train`).

In [ ]:
import sys, glob, warnings
sys.path.insert(0, "..")            # make `src` importable from notebooks/
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

from src import features as ft, evaluate as ev, train as tr, fairness as fr

DATA = sorted(glob.glob("../data/*.xls*") + glob.glob("../data/*.csv"))[0]   # see data/README.md
print("Using", DATA)

In [ ]:
import joblib
from sklearn.base import clone
from src import explain as ex

art = joblib.load("../artifacts/model.joblib")
model, calibrator, thr = art["model"], art["calibrator"], art["threshold"]
C_FN, C_FP = art["c_fn"], art["c_fp"]

df = ft.add_group_labels(ft.clean(ft.load_raw(DATA)))
X, y = ft.get_xy(df)
te_idx, tr_idx = art["test_idx"], art["train_idx"]
X_te, y_te = X.iloc[te_idx].reset_index(drop=True), y.iloc[te_idx].reset_index(drop=True)
d_te = df.iloc[te_idx].reset_index(drop=True)

p_raw = model.predict_proba(X_te)[:, 1]
p_cal = calibrator.predict(p_raw)
flag = (p_cal >= thr).astype(int)
print(f"Threshold {thr:.3f} | flagged {flag.mean():.1%} of test customers | recall {flag[y_te==1].mean():.1%}")

# Part A · Explainability (SHAP)
SHAP is computed on the raw log-odds output. Calibration is monotone, so ranking and direction of effects are unchanged.
*Global* plots show what drives risk in general. *Waterfalls* explain individual decisions, which is what a customer or regulator asks for.

In [ ]:
sv = ex.compute_shap(model, X_te)
ex.plot_global(sv, max_display=15)
ex.top_features(sv, 10)

**Sanity checks to write up:** Do the signs make sense (longer delays → higher risk, higher limit → lower risk)? Do engineered features (utilisation, repayment ratio) show up, or does `PAY_1` dominate? Is anything suspicious, such as a feature that is a near-copy of the label?

In [ ]:
cases = ex.pick_cases(y_te.values, p_cal, thr)
print(cases)
ex.plot_waterfalls(sv, cases, p_cal, y_te)

# Part B · Fairness audit
**Setup.** SEX and MARRIAGE are *not* model inputs. AGE and EDUCATION are inputs, but all four are audited.
The "decision" is the flag (early intervention such as a limit change or payment plan), so a higher flag rate is the burden
group members bear. Metrics: flag rate (demographic parity), TPR (equal opportunity), FPR (with TPR: equalised odds),
PPV (predictive parity) and calibration by group.

In [ ]:
audits = {"SEX_LABEL": "Sex", "AGE_GROUP": "Age group", "EDU_LABEL": "Education", "MARRIAGE_LABEL": "Marital status"}
tables = {}
for col, name in audits.items():
    gm = fr.group_metrics(y_te, flag, p_cal, d_te[col])
    tables[col] = gm
    print(f"\n=== {name} ==="); display(gm.round(3))
    display(fr.disparity_summary(gm).round(3).to_frame("value").T)

## Are the gaps real or noise? Bootstrap CIs
Small groups (e.g. 60+) produce wide intervals. Do not report a gap without its uncertainty.

In [ ]:
for col, name in [("SEX_LABEL", "Sex"), ("AGE_GROUP", "Age group")]:
    ci = fr.bootstrap_group_ci(y_te, flag, p_cal, d_te[col], n_boot=500)
    if col == "AGE_GROUP": ci = ci.reindex(["<30", "30-39", "40-49", "50-59", "60+"])
    fr.plot_group_ci(ci, f"Flag rate / TPR / FPR by {name.lower()} (95% bootstrap CI)", save=f"fairness_{col.lower()}.png")
    display(ci.round(3))

## Does "unawareness" work? Proxy check and a with/without-SEX comparison
Dropping SEX does not stop the model from using correlated features. We (1) test how well the model's inputs predict sex,
and (2) compare group gaps for a model *with* SEX against ours at the **same overall flag rate**.

In [ ]:
print(f"AUC of predicting SEX from the model's input features: {fr.proxy_auc(X_te, d_te['SEX']):.3f}  (0.5 = no proxy information)")

X_s, _ = ft.get_xy(df, drop_protected=False)
m_sex = clone(model).fit(X_s.iloc[tr_idx], y.iloc[tr_idx])
s_sex = m_sex.predict_proba(X_s.iloc[te_idx].reset_index(drop=True))[:, 1]
s_main = p_raw
rows = []
for label, s in [("Without SEX (ours)", s_main), ("With SEX", s_sex)]:
    t = np.quantile(s, 1 - flag.mean())            # same overall flag rate
    f = (s >= t).astype(int)
    gm = fr.group_metrics(y_te, f, s, d_te["SEX_LABEL"])
    rows.append({"model": label, "roc_auc": ev.score_summary(y_te, s)["roc_auc"], "pr_auc": ev.score_summary(y_te, s)["pr_auc"],
                 **fr.disparity_summary(gm)[["flag_rate_ratio (min/max)", "tpr_gap (max-min)", "fpr_gap (max-min)"]].to_dict()})
pd.DataFrame(rows).set_index("model").round(4)

## Trade-offs: what would mitigation cost?
When groups have different base rates, flag-rate parity, equal TPR/FPR and calibration **cannot all hold at once**
(Chouldechova 2017; Kleinberg et al. 2016). Below: the price, in expected cost, of forcing equal TPR across sexes with
group-specific thresholds. This is analysis, **not** a recommendation.

In [ ]:
exp = fr.group_threshold_experiment(y_te, p_cal, d_te["SEX_LABEL"], thr, C_FN, C_FP)
display(exp.round(4)); print("Group thresholds:", {k: round(v, 3) for k, v in exp.attrs["thresholds"].items()})

### Discussion (edit with your own findings)
**What I would do**
* Exclude SEX and MARRIAGE from the model; keep monitoring all four attributes after deployment.
* Report gaps with confidence intervals and tie them to base-rate differences, which are real and may reflect different circumstances.
* Review whether AGE and EDUCATION should be inputs. They may be legally restricted or proxies for something else; the with/without comparison shows what they cost.
* Make the flag a *supportive* intervention (payment plan, outreach) rather than an adverse action where possible: this changes what fairness requires.
* Keep a human in the loop for limit reductions and provide customer-facing reasons (the SHAP waterfalls are the starting point).

**What I would not do**
* Use group-specific thresholds on a protected attribute: that is disparate treatment and generally unlawful for sex in EU credit and insurance pricing.
* Claim the model is "fair" because SEX is removed: see the proxy test.
* Optimise one fairness metric blindly: they conflict, and the choice is a policy decision for risk, legal and compliance.

*(This is analysis, not legal advice. Regulatory details vary by jurisdiction and should be checked with counsel.)*

# Part C · Failure analysis
Where is the model wrong, and why? We look at the most confident errors, error rates by segment, and what distinguishes missed defaulters from caught ones.

In [ ]:
errs = ev.confident_errors(X_te.assign(p_cal=p_cal), y_te.values, p_cal, thr, n=10)
cols = ["p_cal", "y", "LIMIT_BAL", "AGE", "PAY_1", "PAY_2", "PAY_MAX", "UTIL_MEAN", "PAY_RATIO_MEAN", "N_DELAYED"]
print("Most confident FALSE NEGATIVES (defaulted, scored very low):"); display(errs["false_negatives"][cols].round(3))
print("Most confident FALSE POSITIVES (did not default, scored very high):"); display(errs["false_positives"][cols].round(3))

In [ ]:
seg_specs = [("PAY_1", None), ("LIMIT_BAL", [0, 30_000, 60_000, 120_000, 250_000, 1e7]), ("N_DELAYED", None), ("UTIL_MEAN", [-10, 0.1, 0.4, 0.8, 1.0, 10])]
for col, bins in seg_specs:
    print(f"\nError rates by {col}"); display(ev.error_by_segment(X_te, y_te, p_cal, thr, col, bins).round(3))

In [ ]:
fn = (y_te.values == 1) & (p_cal < thr); tp = (y_te.values == 1) & (p_cal >= thr)
print(f"Missed defaulters: {fn.sum()}  |  caught defaulters: {tp.sum()}")
print("Standardised mean differences, missed vs caught defaulters (large |smd| = blind spot):")
ev.compare_groups(X_te, fn, tp, top=10).round(3)

**What to look for / write up**
* Missed defaulters often look like healthy customers until the month they default (sudden shocks). Is that the pattern here: clean PAY history, low utilisation?
* Are false positives mostly customers with long delay histories who recover (revolving/"PAY = 0" behavior)?
* Is the miss rate concentrated in a segment (e.g. very high limits or no recent history)? That is a monitoring item and a data-collection recommendation (income, employment, bureau data).
* SHAP waterfalls for the worst false negatives show *which features misled the model*.